# COMP0178 — Lab 4: PHP and MySQL

Source: [Project Tutorial 4 PDF](../project_tutorials/Project_Tutorial_4.pdf), 32 pages, Dr Yuzuko Nakamura. These are explained notes for the supplied tutorial. Page numbers below refer to PDF pages.

**The point of the lab:** connect a website to a database so a visitor can view existing data and submit new data. The tutorial uses a small chatroom to show the entire path. The code in the slides is an older teaching example; the safer corrections are marked separately below.


## 1. The whole system in one picture — pages 2–3, 20–21

```text
Browser (HTML form and page)
      ↓ request / submitted form
Web server runs PHP
      ↓ SQL query
MySQL stores or retrieves rows
      ↑ query result
PHP builds HTML
      ↑ response
Browser displays the page
```

- **HTML/CSS** controls what the visitor sees. **PHP** runs on the server and decides what to do. **MySQL** stores the data. These are different jobs.
- A database query does not directly appear in the browser. PHP receives the result and writes HTML for the browser.
- The tutorial's basic database sequence is: **connect → make SQL → execute → fetch rows if any → close**.
- `SELECT` reads rows; `INSERT` writes a new row. A form submission normally needs validation before `INSERT`.


## 2. Connect to MySQL — pages 4–6, 10, 29

```php
$connection = mysqli_connect($host, $username, $password, $database);
```

The four arguments are **server, username, password, database name**, in that order. The returned connection is passed to later `mysqli_...` calls. `localhost` means the database server on the same machine. The example chatroom puts this in `database.php` so other pages can reuse it (page 29).

- Use a database account with only the access the application needs. Do not run the website as MySQL `root` (page 5).
- Keep real database passwords out of public web files, Git, and screenshots. The password shown in the lecture is a demonstration value.
- A connection or query can fail. Handle the error; do not silently assume success (page 6).
- Close with `mysqli_close($connection)` when finished (page 10). PHP also closes ordinary connections when their objects are destroyed, but closing promptly is useful when the script continues to do more work.

**Slide correction:** page 6 calls `mysql_error()` after using `mysqli_...`; these are different APIs. For a connection error, use `mysqli_connect_error()` if using return-value error handling. For a query error, use `mysqli_error($connection)`. With PHP 8.1+ default MySQLi settings, errors may instead raise `mysqli_sql_exception`, so use the error approach appropriate to your installation.


## 3. Read rows with SELECT — pages 7–9, 25–26

```php
$query = "SELECT first_name, family_name FROM Users";
$result = mysqli_query($connection, $query);
while ($row = mysqli_fetch_assoc($result)) {
    echo htmlspecialchars($row['first_name'], ENT_QUOTES, 'UTF-8');
}
```

- `$query` is a PHP **string containing SQL**. `mysqli_query` sends that SQL using an open connection.
- A successful `SELECT` returns a result set. `mysqli_fetch_assoc` gets **one row at a time** as an associative array: `$row['first_name']` is one field in the current row.
- The `while` loop keeps fetching until there are no more rows. The slide uses `mysqli_fetch_array`; `mysqli_fetch_assoc` is clearer when you want column names (page 26 uses it).
- PHP may mix ordinary HTML and PHP inside the loop: each fetched row becomes one `<tr>` or `<li>` on the page.
- `htmlspecialchars` is a **safety addition** to the slide: it converts characters such as `<` before user supplied text becomes HTML. It does not change the stored database value.

**Trace:** if the query returns three rows, the loop body runs three times and writes three pieces of HTML. When the next fetch finds no row, the loop stops.


## 4. Write a row with INSERT — pages 11, 27, 30–31

A form such as `<form method="post" action="process.php">` sends input fields to `process.php`. PHP reads them from `$_POST`, checks them, then inserts a row. A successful insert is followed by a redirect back to the page that displays messages.

```text
Visitor enters name and message
  → browser sends POST to process.php
  → PHP checks both fields
  → PHP inserts a message into MySQL
  → PHP redirects to index.php
  → index.php selects rows and displays the new message
```

- SQL `INSERT INTO messages (user, message, time) VALUES (...)` adds one chat message.
- `$_POST['user']` comes from the form field whose `name` is `user`; `$_POST['message']` comes from `name="message"`.
- `$_GET['error']` on `index.php` is used only to display an error passed in the URL after a redirect. It should be escaped before being printed as HTML.
- The slide builds SQL by inserting variables into a string and calls `mysqli_real_escape_string`. **Do not use that pattern for new work.** Use a prepared statement with placeholders for user supplied values:

```php
$stmt = mysqli_prepare($connection,
    'INSERT INTO messages (user, message, time) VALUES (?, ?, ?)');
mysqli_stmt_bind_param($stmt, 'sss', $user, $message, $time);
mysqli_stmt_execute($stmt);
mysqli_stmt_close($stmt);
```

`?` marks values supplied later. `'sss'` says all three values are strings. Prepared statements separate the SQL structure from input values. You still need ordinary input checks such as rejecting an empty name or message.


## 5. Reuse PHP files — pages 12–18

- Put shared database connection code in `database.php` and load it from pages that need it. Put a common header, navigation, and footer in reusable PHP files.
- `include 'file.php'` warns if the file cannot be loaded and the script may continue. `require 'file.php'` stops on a loading failure. Use `require` for essential code such as a database connection or site header.
- `require_once 'file.php'` also prevents a file from being loaded again in the same request. This is useful for files declaring functions/classes (pages 14–15).
- A page can set `$pageTitle = 'Index';` before requiring a shared header. The included header can then use `$pageTitle` because the included code executes in that scope (pages 17–18).
- If a PHP change seems absent in the browser, reload or clear cache before assuming the code failed (page 12).

**Example:** if ten pages share one header file, changing that one file updates the header on all ten pages.


## 6. Follow the chatroom example — pages 21–32

| Part | Pages | What it does |
|---|---:|---|
| Browser interface | 22–23 | Shows messages, a name field, a message field, and a submit button. CSS changes the layout on a narrow screen. |
| `messages` table | 24 | Holds `id`, `user`, `message`, and `time`; `id` is the primary key and auto-increments. |
| `index.php` | 25–27 | Includes the database connection, runs `SELECT * FROM messages ORDER BY id DESC`, prints each row, displays an error if supplied, and contains the form. |
| `database.php` | 29 | Opens the MySQL connection for other PHP files. |
| `process.php` | 30–31 | Reads POST data, checks input, inserts a message, then redirects to `index.php`. |
| CSS | 32 | Controls colours, spacing, width, and small-screen layout. |

`ORDER BY id DESC` puts larger IDs first, so newer messages usually appear at the top. The slides use a very small database to make the data flow easy to see; a real project needs its own tables and relationships from its ER diagram.


## 7. What the tutorial asks you to do this week — page 19

1. Read the feedback on your ER diagram (ERD).
2. Add PHP scripts to **insert, display, or delete** database rows in your project.
3. Connect the parts you have studied: Apache/MAMP/XAMPP style web stack, PHP, SQL, Git, and basic HTML/CSS.

The PDF gives these as weekly TODOs. It does **not** state a submission date on these pages. Use the current Moodle coursework brief for your actual deadline and full project requirements.


## 8. Important corrections before copying slide code

- Pages 6–7 and 11 mix old `mysql_error()` with `mysqli_...`. Use the same MySQLi API throughout.
- Page 11 puts input directly into an SQL string. Pages 30–31 escape input, but prepared statements are the safer default for new PHP code.
- Pages 26–27 echo message/user/error text directly into HTML. Escape output with `htmlspecialchars(..., ENT_QUOTES, 'UTF-8')` to prevent a visitor's text being treated as markup.
- Page 29 shows a literal demo password. Use your own local account and keep credentials out of the public repository.
- The slide snippets have some typography/line-break artifacts; read them as an explanation of the flow, not as a ready-to-run application.

**Official PHP references:** [MySQLi connections](https://www.php.net/manual/en/mysqli.quickstart.connections.php), [prepared statements](https://www.php.net/manual/en/mysqli.quickstart.prepared-statements.php), [MySQLi error modes](https://www.php.net/manual/en/mysqli-driver.report-mode.php), [HTML escaping](https://www.php.net/manual/en/function.htmlspecialchars.php), [require_once](https://www.php.net/manual/en/function.require-once.php).


## 9. SQL practice in this notebook

The next cells use a small **SQLite database in memory** so you can practise `INSERT` and `SELECT` without a configured PHP/MySQL server. The SQL idea is the same; PHP's `mysqli_...` functions themselves require PHP and MySQL. Run the cells from top to bottom. Re-run the setup cell to reset the sample data.


In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
conn.execute('CREATE TABLE messages (id INTEGER PRIMARY KEY, user TEXT, message TEXT, time TEXT)')
conn.executemany(
    'INSERT INTO messages (id, user, message, time) VALUES (?, ?, ?, ?)',
    [(1, 'Cat', 'Hello', '16:26:57'),
     (2, 'Cat', 'Is anyone here?', '17:27:28'),
     (3, 'Ned', 'Hi.', '17:27:37')]
)
print('Sample messages ready:', conn.execute('SELECT COUNT(*) FROM messages').fetchone()[0])


### Practice A — What does `ORDER BY id DESC` do?

Predict the order of the three messages before you run this query. Then compare the result with the chatroom screenshot on page 22.


In [ ]:
rows = conn.execute('SELECT id, user, message FROM messages ORDER BY id DESC').fetchall()
for row in rows:
    print(row)


**Answer check:** the IDs should appear `3, 2, 1`. `DESC` means descending order. This is a SQL ordering rule; PHP only prints the rows it receives.


### Practice B — Add and retrieve a message

Write an `INSERT` for a new message from `Ada` with text `Good morning` at `09:00:00`. Then query the table ordered by descending ID. The new message should appear first. Use parameter placeholders in the Python/SQLite exercise, like the setup cell.


In [ ]:
# Write your INSERT here. Then run a SELECT to check the new row.


**One solution (look after trying):**

```python
conn.execute(
    'INSERT INTO messages (id, user, message, time) VALUES (?, ?, ?, ?)',
    (4, 'Ada', 'Good morning', '09:00:00')
)
print(conn.execute('SELECT id, user, message FROM messages ORDER BY id DESC').fetchall())
```

The first row should be `(4, 'Ada', 'Good morning')`. Re-run the setup cell before trying again so ID 4 does not already exist.


## 10. Revision questions and answers

1. **What does `mysqli_connect` return?** A connection to the selected database, used by later MySQLi calls.
2. **Why is a `while` loop needed after `SELECT`?** Fetching reads one row at a time; the loop handles every row until none remain.
3. **What is the difference between `$_POST` and `$_GET` in this example?** The form sends the new message via POST; the redirect can put a short error message in the URL for the next page to read via GET.
4. **Why use `require_once` for a shared PHP file?** Essential code must load, and `_once` avoids loading it twice in the same request.
5. **Where is a message saved?** In the MySQL `messages` table after `process.php` executes an `INSERT`; the HTML page only displays it.
6. **Why use a prepared statement?** It separates user supplied values from the SQL command structure.
7. **Why escape text when printing it into HTML?** So user supplied characters such as `<` are shown as text instead of being interpreted as HTML.
8. **What is the purpose of the ERD feedback?** It helps you correct the structure of your own project database before adding more PHP operations.


## 11. Exam and project connection

This tutorial is mainly **project implementation**: how a PHP page sends SQL and handles the returned rows. The database ideas also connect to SQL topics in the lecture notes: table columns, primary keys, `SELECT`, `INSERT`, and query results. No particular past-paper question is claimed here. For the project, use your current coursework specification alongside these tutorial notes.
